# Running the Simulations

This notebook explains in detail how to run simulations.

## Activate the Environment

First, activate the `Simulator` environment and import the module.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "Simulator"))
using Simulator

## Simulation Parameters

`SimulationParams` is a structure that encapsulates the parameters of the simulation.

```julia
struct SimulationParams
    "ASV parameters and controller"
    asv::MarineModels.ControlledASV
    "Cable"
    cable::CableModels.Cable
    "Payload"
    payload::PayloadModels.Payload
    "Constraint enforcement gain"
    k_f::Real
end
```

Simulation parameters comprise the parameters of the ASV, the cable, and the payload. The multibody system is simulated as three separate Euler-Lagrange systems with constraints. Constraint enforcement is achieved by using Lagrange multipliers. To prevent numerical drift, we introduce feedback terms with gain `k_f` (for more details, see *Baumgarte stabilization method*).

To create simulation parameters, we first need to define the ASV, cable, and payload.

### ASV

The ASV model is contained in `MarineModels`. The controller is contained in `AdaptiveController`. We need to import both modules:

In [ ]:
using MarineModels
using AdaptiveController

`ControlledASV` is defined as follows:

```julia
struct ControlledASV <: EulerLagrangeX.ControlledEulerLagrangeSystem
    "Physical model of the ASV"
    mdl::ASVModel
    "Controller for the ASV"
    ctrl::Controller
end
```

Therefore, to create a controlled ASV, we need a physical model and a controller.

#### Physical Model

`ASVModel` is defined as follows:

```julia
struct ASVModel <: EulerLagrangeX.EulerLagrangeSystem
    "ASV mass"
    m::Real   
    "Yaw inertia"
    J::Real   
    "Position of the center of mass in the body-fixed coordinate frame"
    xG::Real  
    "Added mass in the surge direction"
    Xu::Real  
    "Added mass in the sway direction"
    Yv::Real  
    "Added mass in yaw"
    Nr::Real  
    "Cross-coupling between sway and yaw"
    Yr::Real  
    "Linear hydrodynamic damping in surge"
    Dul::Real 
    "Linear hydrodynamic damping in sway"
    Dvl::Real 
    "Linear hydrodynamic damping in yaw"
    Drl::Real 
    "Quadratic hydrodynamic damping in surge"
    Duq::Real 
    "Quadratic hydrodynamic damping in sway"
    Dvq::Real 
    "Quadratic hydrodynamic damping in yaw"
    Drq::Real 
    "Ocean current"
    V::Union{Rn, Function}
    "Perturbation"
    μ::Union{Rn, Function}
end
```

Ocean current `V` can be either a 2D vector or a function that accepts one scalar parameter -- time -- and returns a 2D vector.
Perturbation `μ` can be either a 2D vector or a function that accepts two parameters -- time and position -- and returns a 2D vector.

Let's create an ASV model:

In [ ]:
m = 1000.0    # mass
I_z = 4500.0  # moment of inertia
xG = 2.0      # x-coordinate of the center of gravity
Xu = 500.0    # added mass in surge
Yv = 2000.0   # added mass in sway
Nr = 1500.0   # added mass in yaw
Yr = 0.0      # cross coupling added mass

Dul = 500.0   # linear drag in surge
Dvl = 2000.0  # linear drag in sway
Drl = 4000.0  # linear drag in yaw
Duq = Dul / 2 # quadratic drag in surge
Dvq = Dvl / 2 # quadratic drag in sway
Drq = Drl / 2 # quadratic drag in yaw

V_c = [0.5, 0.5] # current velocity
μ = [0.0, 0.0]   # perturbation

# Create the ASV model
asv_model = ASVModel(m, I_z, xG, Xu, Yv, Nr, Yr, Dul, Dvl, Drl, Duq, Dvq, Drq, V_c, μ)

#### Controller

We use the `ASVAdaptiveController` from the `AdaptiveController` module. The structure is defined as follows:

```julia
struct ASVAdaptiveController <: MarineModels.Controller
    "Line-of-sight parameters"
    los::LOSParameters
    "Desired path"
    path::Function
    "Adaptive linearizing controller parameters"
    ctrl::AdaptiveLinearizingController
    "Heading control proportional gain"
    k_ψ::Real
    "Heading control derivative gain"
    k_r::Real
end
```

Line-of-sight parameters consit of the speed `U` and lookahead distance `Δ`. 
Path is a function that takes a scalar parameter and returns the 2D reference position.
The adaptive linearizing controller has three parameters: velocity feedback gain, `k_v`, adaptation gain, `k_a`, and control point coefficient, `ε`. Note that `ε` must be positive and strictly less than one.
Heading controller is a PD controller (that outputs yaw torque) to align the ASV with the line-of-sight heading.

Let's create the controller:

In [ ]:
# LOS parameters
U = 3.0 # desired speed
Δ = 5.0 # lookahead distance
los = LOSParameters(U, Δ)

# Path
function circular_path(s::Real)
    R = 100.0 # radius of the circle
    ϕ0 = -π / 2 # initial phase
    p_o = [20.0, 100.0] # center of the circle
    return R * [cos(s + ϕ0), sin(s + ϕ0)] + p_o
end

# Adaptive controller parameters
k_v = 0.5 # velocity gain
k_a = 5.0 # adaptation gain
ε = 0.7   # control point coefficient
adaptive_controller = AdaptiveLinearizingController(k_v, k_a, ε)

# Heading control parameters
k_ψ = 1.0e4 # heading control gain
k_r = 2.0e4 # yaw rate control gain

asv_controller = ASVAdaptiveController(los, circular_path, adaptive_controller, k_ψ, k_r)

Now, we can assemble the physical parameters and the controller into a controlled ASV structure:

In [ ]:
asv = ControlledASV(asv_model, asv_controller)

### Cable

`Cable` is a structure defined in the `CableModels` package:

```julia
struct Cable <: EulerLagrangeX.EulerLagrangeSystem
    "Number of cable segments"
    N::Integer
    "Length of cable segments"
    L::Union{Real, Rn}
    "Mass of cable segments"
    m::Union{Real, Rn}
    "Longitudinal damping"
    c_u::Union{Real, Rn}
    "Lateral damping"
    c_s::Union{Real, Rn}
    "Fluid flow velocity (ocean current) (a 2D vector or a function that accepts one scalar - time - and returns a 2D vector)"
    V::Union{Rn, Function}
    "Perturbation (a 2D vector or a function that accepts a scalar and a 2D vector - time and position - and returns a 2D vector)"
    μ::Union{Rn, Function}
end
```

Let's create an 8-segment, 100m long cable:

In [ ]:
using CableModels

N = 8 # segments
L = 100.0 / N # length of each segment
ϱ = 0.5 # linear density
m = ϱ * L # mass of each segment
c_u = 0.25 # longitudinal damping coefficient
c_s = 0.25 # lateral damping coefficient

cable = Cable(N, L, m, c_u, c_s, V_c, μ)

### Payload

`Payload`, defined in `PayloadModels`, has the following parameters:

```julia
struct Payload <: EulerLagrangeX.EulerLagrangeSystem
    "Mass of the payload"
    m::Real
    "Linear damping coefficient"
    d_l::Real
    "Quadratic damping coefficient"
    d_q::Real
    "Fluid flow velocity (ocean current) (a 2D vector or a function that accepts one scalar - time - and returns a 2D vector)"
    V::Union{Rn, Function}
    "Perturbation (a 2D vector or a function that accepts a scalar and a 2D vector - time and position - and returns a 2D vector)"
    μ::Union{Rn, Function}
end
```

In [ ]:
using PayloadModels

m = 250.0 # mass of the payload
d_l = 250.0 # linear damping coefficient
d_q = 0.0 # quadratic damping coefficient

payload = Payload(m, d_l, d_q, V_c, μ)

Now, we can assemble everything into `SimulationParams`:

In [ ]:
k_f = 1.0 
sim_params = SimulationParams(asv, cable, payload, k_f)

### Default parameters

We can create default simulation parameters (i.e., the ones used in the CEP paper) using the function

```julia
default_simulation(; N=8, V_c=default_constant_current(), μ=zeros(2), k_f=1.0)
```

In [ ]:
sim_params_default = default_simulation()

## State Vector

The state vector of the system is defined as:

$$ x = {\rm col} \left( q_{\rm ASV}, q_{\rm cable}, q_{\rm payload}, \dot{q}_{\rm ASV}, \dot{q}_{\rm cable}, \dot{q}_{\rm payload}, x_{\rm ctrl} \right) $$

where $q_{\rm ASV} = [x_{\rm ASV}, y_{\rm ASV}, \psi]$ are the generalized coordinates of the ASV, $q_{\rm cable} = [x_0, y_0, \theta_1, \ldots, \theta_N]$ are the generalized coordinates of the cable, where $x_0, y_0$ is the position of the first link and $\theta_i$ are the angles of the links, $q_{\rm payload} = [x_p, y_p]$ is the position of the payload, $\dot{q}$ are the generalized velocities, and $x_{\rm ctrl} = {\rm col}(s, \hat{\zeta})$ is the state of the ASV controller, consisting of the path parameter, $s$, and the estimates of the adaptive controller, $\zeta$.

To make working with the state vector easier, we can use the `SimulationState` struct:

```julia
struct SimulationState
    "State of the ASV (position, velocity, internal states)"
    asv_state::Tuple{Rn, Rn, Rn}
    "State of the cable model (position, velocity)"
    cable_state::Tuple{Rn, Rn}
    "State of the payload model (position, velocity)"
    payload_state::Tuple{Rn, Rn}
end
```

For conversion, use the functions

```julia
get_simulation_state(x::Vector, sim::SimulationParams) -> SimulationState

vectorize(state::SimulationState) -> Vector
```

### Constraints

When designing an initial state for the simulation, be aware of the kinematic constraints in the system.
There are two constraints:

1. The first link of the cable is attached to the ASV, i.e., $[x_{\rm ASV}, y_{\rm ASV}] = [x_0, y_0]$.
2. The other end of the cable is attached to the payload, i.e., $[x_p, y_p] = [x_0, y_0] + \sum_i L_i [\cos\theta_i, \sin\theta_i]$

Let's create an intial feasible state:

In [ ]:
# Coordinates
q_ASV = zeros(3)
p0_cable = q_ASV[1:2] # initial position of the cable (same as ASV)
θ_cable = range(-π, 0, length=cable.N) # initial angles of the cable segments
q_cable = [p0_cable; θ_cable] # initial state of the cable
q_payload = p0_cable + [sum(cable.L .* cos.(θ_cable)), sum(cable.L .* sin.(θ_cable))] # initial position of the payload

# Velocities
q_dot_ASV = [1.0, 0.0, 0.0] # initial velocity of the ASV
v0_cable = q_dot_ASV[1:2] # initial velocity of the cable (same as ASV)
ω_cable = zeros(cable.N) # initial angular velocities of the cable segments
q_dot_cable = [v0_cable; ω_cable] # initial state of the cable
q_dot_payload = v0_cable + [sum(cable.L .* -sin.(θ_cable) .* ω_cable), sum(cable.L .* cos.(θ_cable) .* ω_cable)] # initial velocity of the payload

### Parameter Vector

To initialize the adaptive controller, we can use the following function:

```julia
parameter_vector(param::LinkedMassesParameters, ε::Real, V_c::Rn=zeros(2)) -> Vector
```

where `LinkedMassesParameters` is a struct defined as

```julia
struct LinkedMassesParameters
    "Link length"
    L::Real
    "First mass"
    m0::Real
    "Second mass"
    m::Real
    "First damping coefficient"
    c0::Real
    "Second damping coefficient"
    c::Real
end
```

Now, suppose that the estimated parameters of the system are:

In [ ]:
# Path parameter
s = 0.0 # initial path parameter

# Adaptive controller state
L_hat = cable.L * cable.N # total length of the cable
m0_hat = asv_model.m # vehicle mass estimate
m_hat = 2 * payload.m # payload mass estimate
c0_hat = 500.0 # damping coefficient estimate
c_hat = 0.5 * payload.d_l # damping coefficient estimate
V_c_hat = zeros(2) # current velocity estimate

linked_masses_estimate = LinkedMassesParameters(L_hat, m0_hat, m_hat, c0_hat, c_hat)

ζ = parameter_vector(linked_masses_estimate, ε, V_c_hat)

x_ASV = [s; ζ]

When put together, the state of the system is:

In [ ]:
sim_state = SimulationState((q_ASV, q_dot_ASV, x_ASV), (q_cable, q_dot_cable), (q_payload, q_dot_payload))

And the corresponding vector is:

In [ ]:
sim_state_vector = vectorize(sim_state)

### Default Initial State

To generate the default initial state used in the CEP paper, use

```julia
default_initial_state(; N_cable=8) -> SimulationState
```

In [ ]:
sim_state_default = default_initial_state()

## Running the Simulation

The ODEs of the whole system are implemented in

```julia
closed_loop_ode(x::Vector, sim::SimulationParams, t::Real) -> Vector
```

We can use the `DifferentialEquations` package to solve the ODEs. Use `ODEProblem` and `solve`:

In [ ]:
using DifferentialEquations

prob = ODEProblem(closed_loop_ode, sim_state_vector, (0.0, 200.0), sim_params)
sol = solve(prob, Tsit5())

## Plotting the Results

### Errors

We can use the `ClosedLoopState` struct and the corresponding `get_closed_loop_state` function to get the path-following errors, velocity errors, and angular velocities of the cable at a given time. Syntax:

```julia
struct ClosedLoopState
    "Path-following error x"
    e_x::Real
    "Path-following error y"
    e_y::Real
    "Velocity error x"
    v_x::Real
    "Velocity error y"
    v_y::Real
    "Cable angular velocities"
    θ_dot::Rn
end

get_closed_loop_state(x::Vector, sim::SimulationParams) -> ClosedLoopState
```

The code snippet below obtains and plots the error variables:

In [ ]:
# Preallocate
e_x = zeros(length(sol.u))
e_y = zeros(length(sol.u))
v_x = zeros(length(sol.u))
v_y = zeros(length(sol.u))
θ_dot = zeros((length(sol.u), cable.N))

# Get closed-loop states
for (i, x) in enumerate(sol.u)
    cls = get_closed_loop_state(x, sim_params)

    e_x[i] = cls.e_x
    e_y[i] = cls.e_y
    v_x[i] = cls.v_x
    v_y[i] = cls.v_y
    θ_dot[i, :] = cls.θ_dot
end

# Plotting
using Plots

plt_path_error = plot(sol.t, e_x, label="e_x", xlabel="Time (s)", ylabel="Path Error (m)", title="Path Following Error")
plot!(plt_path_error, sol.t, e_y, label="e_y")
plt_velocity = plot(sol.t, v_x, label="v_x", xlabel="Time (s)", ylabel="Velocity (m/s)", title="Velocity Error")
plot!(plt_velocity, sol.t, v_y, label="v_y")
θ_dot_labels = reshape(["θ_dot_$i" for i in 1:cable.N], (1, cable.N))
plt_θ_dot = plot(sol.t, θ_dot, label=θ_dot_labels, xlabel="Time (s)", ylabel="Angular Rate (rad/s)", title="Cable Segment Angular Velocities")
plot(plt_path_error, plt_velocity, plt_θ_dot, layout=@layout([[a b]; c]), size=(900, 600))

### Trajectories

To obtain the configuration of the system at a given time, use the `PlottingState` struct and the corresponding `get_plotting_state` function:

```julia
struct PlottingState
    "ASV coordinates"
    q_asv::Rn
    "Cable knot x-coordinates"
    x_cable::Rn
    "Cable knot y-coordinates"
    y_cable::Rn
    "Payload coordinates"
    q_payload::Rn
    "Control point"
    p_ε::Rn
    "Path reference"
    p_path::Rn
end

get_plotting_state(x::Vector, sim::SimulationParams) -> PlottingState
```

To plot the configuration, we can invoke the `plot_towed_system!` function:

```julia
plot_towed_system!(plt::Plots.Plot, pls::PlottingState, dim::ASVDimensions, color) -> Plots.Plot
```

The ASV is represented by a polygon. The shape of the polygon looks like this:

```
  /\    ->  this is the "bow"
 /  \
 |  |
 |  |   -> this is the "body"
 |__|
```

The parameters of the polygon are specified in the `ASVDimensions` struct:

```julia
struct ASVDimensions
    "Body length"
    L_body::Real
    "Bow length"
    L_bow::Real
    "Body width"
    W_body::Real
end
```

The code snippet below plots the configuration of the system at a given time:
 

In [ ]:
using Printf
dims = ASVDimensions(20.0, 6.0, 8.0)

function plot_configuration(t::Real)
    # Get the state at time t
    x = sol(t)
    pls = get_plotting_state(x, sim_params)

    # Path plot
    arg = range(0, 2π, length=100)
    p_path = hcat(circular_path.(arg)...)
    title_string = @sprintf("Trajectory at t = %.2f s", t)
    plt = plot(p_path[2, :], p_path[1, :], label="Path", xlabel="East (m)", ylabel="North (m)", title=title_string, color=:black, aspect_ratio=:equal)

    # Towed system plot
    plot_towed_system!(plt, pls, dims, :blue)
    return plt
end

plot_configuration(0.0)

The square represents the current path point, the cross is the control point, and the regular pentagon is the payload.

Now, we can create an animation and save it as a gif:

In [ ]:
fps = 20.0 # frames per second
spd = 5.0 # playback speed
Ts = spd / fps # time step between frames
animation = @animate for t in 0:Ts:sol.t[end]
    plot_configuration(t)
end
gif(animation, "towed_system_simulation.gif", fps=fps)